<a href="https://colab.research.google.com/github/Robingif/cognitive-assignments/blob/main/Copy_of_UCS420_LA9_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [ ]:
ROLL_NO = 1024170017         # <-- your full roll number
NAME = "Robin"            # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [ ]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [ ]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [ ]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

Roll number: 1024170017   Name: Robin   Fingerprint: F938970A   Tickets: 14

Ticket_ID                                                                              Ticket_Text
      T01                 I reinstalled the Python since yesterday. The same problem occurs again.
      T02                      WiFi is not working after resetting my password. Please resolve it.
      T03          Server is not working when I use mobile hotspot. The same problem occurs again.
      T04                       I was locked out of the Matlab since yesterday. Please resolve it.
      T05                                                    I need Windows 11 by 5.30 p.m. today.
      T06             I cannot access the email before the assignment deadline. Please resolve it.
      T07                                                          Is it fixed?? Pls reply ASAP :(
      T08 I tried restarting the student portal when connected to campus Wi-Fi. Please resolve it.
      T09     I keep getting log

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [ ]:
# Your code
results = []

for _, row in df.iterrows():
    processed = process_ticket(row["Ticket_Text"])
    results.append({
        "Ticket_ID": row["Ticket_ID"],
        "sentences": len(processed["sentences"]),
        "tokens": len(processed["tokens"]),
        "punctuation tokens": len(processed["punct"]),
        "stopwords": len(processed["stops"]),
        "unique tokens": len(set(processed["tokens"])),
        "final tokens": len(processed["final"])
    })

# Create table
result_df = pd.DataFrame(results)

# TOTAL row
total_row = {
    "Ticket_ID": "TOTAL",
    "sentences": result_df["sentences"].sum(),
    "tokens": result_df["tokens"].sum(),
    "punctuation tokens": result_df["punctuation tokens"].sum(),
    "stopwords": result_df["stopwords"].sum(),
    "unique tokens": result_df["unique tokens"].sum(),
    "final tokens": result_df["final tokens"].sum()
}

result_df = pd.concat(
    [result_df, pd.DataFrame([total_row])],
    ignore_index=True
)

result_df


,Ticket_ID,sentences,tokens,punctuation tokens,stopwords,unique tokens,final tokens
0,T01,2,13,2,5,11,6
1,T02,2,13,2,5,12,6
2,T03,2,16,2,7,15,7
3,T04,2,14,2,6,13,6
4,T05,1,9,1,2,9,6
5,T06,2,15,2,7,13,6
6,T07,2,10,4,2,9,4
7,T08,2,16,2,5,15,9
8,T09,2,17,2,7,16,8
9,T10,1,9,1,3,9,5


**Observe:** Which final tokens in your data are not real words (e.g. n't, 's, ca)? Where do they come from?

*Your answer:*

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [ ]:

import re
import pandas as pd

# =========================
# TEST LINES
# =========================

TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]


# =========================
# FINDALL PATTERNS
# =========================

# Words longer than 5 letters
long_words_pattern = r"\b[A-Za-z]{6,}\b"

# Numbers:
# 2.0.1, 95.5, 5.30 and ordinary integers
numbers_pattern = r"\b\d+(?:\.\d+)+\b|\b\d+\b"

# Capitalized words
capitalized_pattern = r"\b[A-Z][A-Za-z]*\b"

# Words starting with a vowel
vowel_words_pattern = r"\b[AEIOUaeiou][A-Za-z]*\b"


# =========================
# RUN FINDALL ON TICKETS
# =========================

regex_results = []

for _, row in df.iterrows():

    text = row["Ticket_Text"]

    regex_results.append({
        "Ticket_ID": row["Ticket_ID"],
        "long_words": re.findall(long_words_pattern, text),
        "numbers": re.findall(numbers_pattern, text),
        "capitalized_words": re.findall(capitalized_pattern, text),
        "vowel_words": re.findall(vowel_words_pattern, text)
    })

regex_df = pd.DataFrame(regex_results)

regex_df

,Ticket_ID,long_words,numbers,capitalized_words,vowel_words
0,T01,"[reinstalled, Python, yesterday, problem, occurs]",[],"[I, Python, The]","[I, occurs, again]"
1,T02,"[working, resetting, password, Please, resolve]",[],"[WiFi, Please]","[is, after, it]"
2,T03,"[Server, working, mobile, hotspot, problem, oc...",[],"[Server, I, The]","[is, I, use, occurs, again]"
3,T04,"[locked, Matlab, yesterday, Please, resolve]",[],"[I, Matlab, Please]","[I, out, of, it]"
4,T05,[Windows],"[11, 5.30]","[I, Windows]",[I]
5,T06,"[cannot, access, before, assignment, deadline,...",[],"[I, Please]","[I, access, email, assignment, it]"
6,T07,[],[],"[Is, Pls, ASAP]","[Is, it, ASAP]"
7,T08,"[restarting, student, portal, connected, campu...",[],"[I, Wi, Fi, Please]","[I, it]"
8,T09,"[getting, logged, resetting, password, Please,...",[],"[I, VPN, Please]","[I, out, of, after, it]"
9,T10,[],"[91, 98765, 43210, 0175, 239, 3201]",[Call],"[on, or]"


**Observe:** How many of your capitalized words are capitalized only because they start a sentence?

*Your answer:*

In [ ]:
# =========================
# REDACTION PATTERNS
# =========================

# Email
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

# URL
url_pattern = r"\b(?:https?://|www\.)[^\s]+"

# Phone numbers
phone_pattern = r"(?<!\d)(?:\+\d{1,3}[-\s]?)?\d{3,5}[-\s]\d{3,5}(?:[-\s]\d{3,5})?(?!\d)"


def redact_ticket(text):

    # URLs first so anything inside a URL isn't affected
    text = re.sub(url_pattern, "<URL>", text)

    # Emails
    text = re.sub(email_pattern, "<EMAIL>", text)

    # Phone numbers
    text = re.sub(phone_pattern, "<PHONE>", text)

    return text


redacted_results = []

for _, row in df.iterrows():

    redacted_results.append({
        "Ticket_ID": row["Ticket_ID"],
        "original": row["Ticket_Text"],
        "redacted": redact_ticket(row["Ticket_Text"])
    })

redacted_df = pd.DataFrame(redacted_results)

redacted_df

,Ticket_ID,original,redacted
0,T01,I reinstalled the Python since yesterday. The ...,I reinstalled the Python since yesterday. The ...
1,T02,WiFi is not working after resetting my passwor...,WiFi is not working after resetting my passwor...
2,T03,Server is not working when I use mobile hotspo...,Server is not working when I use mobile hotspo...
3,T04,I was locked out of the Matlab since yesterday...,I was locked out of the Matlab since yesterday...
4,T05,I need Windows 11 by 5.30 p.m. today.,I need Windows 11 by 5.30 p.m. today.
5,T06,I cannot access the email before the assignmen...,I cannot access the email before the assignmen...
6,T07,Is it fixed?? Pls reply ASAP :(,Is it fixed?? Pls reply ASAP :(
7,T08,I tried restarting the student portal when con...,I tried restarting the student portal when con...
8,T09,I keep getting logged out of the VPN after res...,I keep getting logged out of the VPN after res...
9,T10,Call me on +91 98765 43210 or 0175-239-3201.,Call me on <PHONE> or <PHONE>.


In [ ]:
# =========================
# TEST THE PATTERNS
# =========================

for line in TEST_LINES:

    print("\n" + "=" * 80)
    print("ORIGINAL:", line)

    print("Long words:",
          re.findall(long_words_pattern, line))

    print("Numbers:",
          re.findall(numbers_pattern, line))

    print("Capitalized words:",
          re.findall(capitalized_pattern, line))

    print("Vowel-starting words:",
          re.findall(vowel_words_pattern, line))

    print("Redacted:",
          redact_ticket(line))


ORIGINAL: Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.
Long words: ['helpdesk', 'thapar', 'thapar', 'thapar']
Numbers: []
Capitalized words: ['Mail']
Vowel-starting words: ['it', 'edu', 'or', 'a', 'univ', 'ac', 'in', 'edu', 'or', 'edu', 'it']
Redacted: Mail <EMAIL> or <EMAIL>; visit <URL> or <URL>

ORIGINAL: Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.
Long words: ['Version']
Numbers: ['91', '98765', '43210', '91', '98765', '43210', '0175', '239', '3201', '2.0.1', '95.5', '1', '250']
Capitalized words: ['Call', 'Version', 'Rs']
Vowel-starting words: ['or', 'is']
Redacted: Call <PHONE>, <PHONE> or <PHONE>. Version 2.0.1 is 95.5% done; fee Rs 1,250.

ORIGINAL: The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.
Long words: []
Numbers: ['5.30']
Capitalized words: ['The']
Vowel-starting words: ['of', 'art', 'isn', 'open', 'in', 'at']
Redacted: The state-

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [ ]:
# Your code

**Observe:** How many of your tickets are tokenized differently by my_tokenize and word_tokenize?

*Your answer:*

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
# Your code

**Observe:** Give one over-stemming and one under-stemming example from your table.

*Your answer:*

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
# Your code

**Observe:** Why does TTR change from S1 to S4 in your data?

*Your answer:*